# 22.6 — Adversarial search (minimax, alpha-beta)

Minimax chooses the move whose worst opponent reply is best; alpha-beta skips branches that cannot matter.

Games add an opponent who selects the successor you like least. Tree search still applies, but values now alternate between max layers and min layers. MCTS scales this idea when exact minimax is too large.

Save a copy to Drive to edit.

In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt

random.seed(22)


@dataclass
class GameNode:
    name: str
    player: str
    children: List[str]
    value: Optional[float] = None
    features: Optional[Tuple[float, float]] = None


def evaluate_leaf(node, weights=(2, -1)):
    if node.value is not None:
        return node.value
    if node.features is None:
        raise ValueError("leaf needs value or features")
    return node.features[0] * weights[0] + node.features[1] * weights[1]


def minimax(tree, node_name, depth=None, weights=(2, -1), stats=None):
    if stats is None:
        stats = {"leaves": 0, "nodes": 0}
    stats["nodes"] += 1
    node = tree[node_name]
    stop_for_depth = depth is not None and depth == 0
    if not node.children or stop_for_depth:
        stats["leaves"] += 1
        return evaluate_leaf(node, weights), node_name
    child_values = []
    next_depth = None if depth is None else depth - 1
    for child in node.children:
        child_value, _ = minimax(tree, child, next_depth, weights, stats)
        child_values.append((child_value, child))
    if node.player == "max":
        best_value, best_child = max(child_values)
        return best_value, best_child
    best_value, best_child = min(child_values)
    return best_value, best_child


def alpha_beta(tree, node_name, depth=None, alpha=-math.inf, beta=math.inf, order=None, weights=(2, -1), stats=None):
    if stats is None:
        stats = {"leaves": 0, "nodes": 0, "pruned": 0, "visited": []}
    stats["nodes"] += 1
    node = tree[node_name]
    stop_for_depth = depth is not None and depth == 0
    if not node.children or stop_for_depth:
        stats["leaves"] += 1
        stats["visited"].append(node_name)
        return evaluate_leaf(node, weights)
    children = list(node.children)
    if order and node_name in order:
        wanted = order[node_name]
        children = sorted(children, key=lambda child: wanted.index(child) if child in wanted else len(wanted))
    next_depth = None if depth is None else depth - 1
    if node.player == "max":
        value = -math.inf
        for index, child in enumerate(children):
            value = max(value, alpha_beta(tree, child, next_depth, alpha, beta, order, weights, stats))
            alpha = max(alpha, value)
            if alpha >= beta:
                stats["pruned"] += len(children) - index - 1
                break
        return value
    value = math.inf
    for index, child in enumerate(children):
        value = min(value, alpha_beta(tree, child, next_depth, alpha, beta, order, weights, stats))
        beta = min(beta, value)
        if alpha >= beta:
            stats["pruned"] += len(children) - index - 1
            break
    return value


def build_binary_game(depth, branching=2, misleading=False, heuristic=False):
    tree = {}
    leaf_index = 0

    def make_node(prefix, level, player):
        nonlocal leaf_index
        if level == depth:
            base = ((leaf_index * 7 + depth * 3) % 17) - 5
            if misleading and prefix.endswith("0"):
                base += 12
            if heuristic:
                tree[prefix] = GameNode(prefix, player, [], features=(base % 5 + 1, leaf_index % 4))
            else:
                tree[prefix] = GameNode(prefix, player, [], value=base)
            leaf_index += 1
            return prefix
        children = []
        for branch in range(branching):
            child = f"{prefix}.{branch}"
            children.append(make_node(child, level + 1, "min" if player == "max" else "max"))
        tree[prefix] = GameNode(prefix, player, children)
        return prefix

    make_node("root", 0, "max")
    return tree


def adversarial_ladder():
    return [
        {"name": "D1 four-leaf lesson", "tree": lesson_tree(), "depth": None, "order": {"root": ["A", "B"], "B": ["B1", "B2"]}},
        {"name": "D2 wider ordered", "tree": build_binary_game(3, 3), "depth": None, "order": None},
        {"name": "D3 deeper poor order", "tree": build_binary_game(5, 2), "depth": None, "order": None},
        {"name": "D4 depth-limited heuristic", "tree": build_binary_game(4, 3, heuristic=True), "depth": 3, "order": None},
        {"name": "D5 misleading high leaves", "tree": build_binary_game(5, 3, misleading=True), "depth": None, "order": None},
    ]


def lesson_tree():
    return {
        "root": GameNode("root", "max", ["A", "B"]),
        "A": GameNode("A", "min", ["A1", "A2"]),
        "B": GameNode("B", "min", ["B1", "B2"]),
        "A1": GameNode("A1", "leaf", [], value=3),
        "A2": GameNode("A2", "leaf", [], value=5),
        "B1": GameNode("B1", "leaf", [], value=2),
        "B2": GameNode("B2", "leaf", [], value=9),
    }

## The concept, built once (D1)

For a max node, $V(s)=\max_a V(T(s,a))$; for a min node, $V(s)=\min_a V(T(s,a))$. The lesson tree has leaves $[[3,5],[2,9]]$.

In [ ]:
tree = lesson_tree()
stats = {"leaves": 0, "nodes": 0}
root_value, chosen_child = minimax(tree, "root", stats=stats)
print("root value", root_value, "chosen move", chosen_child)
assert min(3, 5) == 3
assert min(2, 9) == 2
assert root_value == 3
assert chosen_child == "A"

Alpha-beta keeps the same backed-up value but prunes when $\alpha\ge\beta$. With good ordering, the lesson visits $3$ of $4$ leaves and prunes $1$. Depth-limited evaluation uses $[x_1,x_2]\cdot[2,-1]$.

In [ ]:
ab_stats = {"leaves": 0, "nodes": 0, "pruned": 0, "visited": []}
ordering = {"root": ["A", "B"], "B": ["B2", "B1"]}
ab_value = alpha_beta(tree, "root", order=ordering, stats=ab_stats)
dot_products = [evaluate_leaf(GameNode("e", "leaf", [], features=f)) for f in [(2, 1), (1, 3), (3, 0)]]
print("alpha-beta value", ab_value, "visited", ab_stats["leaves"], "pruned", ab_stats["pruned"])
print("dot products", dot_products)
assert ab_value == 3
assert ab_stats["leaves"] == 3
assert 4 - ab_stats["leaves"] == 1
assert dot_products == [3, -1, 6]

## The dataset ladder

D1–D5 are deterministic game trees with increasing depth, branching, heuristic cutoffs, and misleading high leaves.

In [ ]:
ladder = adversarial_ladder()
for rung in ladder:
    leaves = [name for name, node in rung["tree"].items() if not node.children]
    print(rung["name"], "nodes", len(rung["tree"]), "leaves", len(leaves), "depth", rung["depth"])
    print("sample leaves", leaves[:5])

## Run the SAME method across D1–D5

Metric: backed-up value error relative to full minimax when available, plus leaves evaluated.

In [ ]:
results = []
for rung in ladder:
    exact_stats = {"leaves": 0, "nodes": 0}
    exact_value = minimax(rung["tree"], "root", stats=exact_stats)[0]
    ab_stats = {"leaves": 0, "nodes": 0, "pruned": 0, "visited": []}
    ab_value = alpha_beta(rung["tree"], "root", depth=rung["depth"], order=rung["order"], stats=ab_stats)
    error = abs(exact_value - ab_value)
    results.append({"name": rung["name"], "value": ab_value, "exact": exact_value, "error": error, "leaves": ab_stats["leaves"], "nodes": ab_stats["nodes"]})
print("rung | value | exact | error | leaves | nodes")
for row in results:
    print(row["name"], row["value"], row["exact"], row["error"], row["leaves"], row["nodes"])

## Results visualization

Left panels show evaluated versus pruned leaf counts. The summary curve tracks error and leaves as the trees grow.

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(16, 3))
for axis, row in zip(axes, results):
    axis.bar(["visited", "error"], [row["leaves"], row["error"]], color=["steelblue", "tomato"])
    axis.set_title(row["name"].split()[0])
plt.show()
sizes = list(range(1, len(results) + 1))
fig, axis = plt.subplots(figsize=(6, 3))
axis.plot(sizes, [row["leaves"] for row in results], marker="o", label="leaves")
axis.plot(sizes, [row["error"] for row in results], marker="s", label="value error")
axis.set_xlabel("D rung")
axis.legend()
plt.show()

## Pitfall on D5: maximizing the best leaf

A tempting leaf can be unreachable under optimal opponent play. The fix is to back up through min layers, and then improve ordering for efficiency rather than changing semantics.

In [ ]:
d5 = ladder[-1]
tempting_leaf = max(evaluate_leaf(node) for node in d5["tree"].values() if not node.children)
proper_value = minimax(d5["tree"], "root")[0]
bad_gap = tempting_leaf - proper_value
print("tempting best leaf", tempting_leaf)
print("proper minimax value", proper_value)
print("overstatement", bad_gap)
assert bad_gap >= 0

## Evaluate it + Practice

- Metric: value error and leaves evaluated.
- No-skill baseline: choose the child containing the largest leaf.
- Cheap sanity check: D1 root value is exactly 3 and alpha-beta visits exactly 3 leaves.
- Ablation: turn off min backups and the D5 value is overstated.
- Failure signals: alpha-beta value changes, or depth-limited error stays large.

### Practice
Change D5 move ordering and compare leaf counts without changing value.

### Practice
Add a new feature vector to D4 and inspect depth-limited error.

### Practice
Plot pruning rate instead of raw leaf visits.